# Tuning extendido: LightGBM + target encoding + class_weight

Este notebook busca mejorar el mejor modelo de la baseline (LightGBM) en el escenario extendido. Todo se selecciona en **validación**; el test se usa una sola vez al final.

**Estrategia:**

1. `RandomizedSearchCV` con `StratifiedKFold(3)` sobre una submuestra estratificada de train (para que sea viable en local).
2. Comparación de cuatro variantes con los mejores hiperparámetros: One-Hot, One-Hot agrupando categorías raras (`min_frequency=0.01`), target encoding con validación cruzada interna y `class_weight='balanced'`.
3. Selección por `f1_macro` en validación; evaluación final en test.

**Target encoding:** `TargetEncoderCV` de `src/preprocessing_features.py` calcula las medias de clase *out-of-fold* en train. Así el modelo no ve el target de la fila que codifica y se evita la fuga típica del target encoding.

**Nota:** las clases ya están balanceadas (~25 % cada una), por lo que `class_weight` probablemente no ayude; se prueba para documentarlo.

In [ ]:
import sys
sys.path.append('../..')

import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

from src.model_features import (
    GRID_LGBM_EXT,
    PARAMS_EXT,
    construir_lgbm_features,
    evaluar_modelo_ordinal,
    metricas_a_fila,
)
from src.preprocessing_features import FEATURES_EXT_PATH, preparar_dataset_features
from src.utils import (
    CLASES,
    REPORTS_DIR,
    SEED,
    cargar_dataset,
    fijar_semilla,
    guardar_modelo,
    split_estratificado,
)

sns.set_theme(style='whitegrid')
fijar_semilla(SEED)
print('Semilla:', SEED)

In [ ]:
df = preparar_dataset_features(
    cargar_dataset(FEATURES_EXT_PATH),
    descartar_baseline=False,
)
X_train, X_val, X_test, y_train, y_val, y_test = split_estratificado(
    df,
    target='price_range',
    excluir=['price_clean', 'id'],
    seed=SEED,
)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

# Submuestra estratificada solo para la búsqueda; el ajuste final usa todo train.
X_busqueda, _, y_busqueda, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(25000, len(X_train)),
    stratify=y_train,
    random_state=SEED,
)
print('Submuestra de búsqueda:', X_busqueda.shape)

In [ ]:
base = construir_lgbm_features(
    X_busqueda,
    imputar=True,
    **PARAMS_EXT['lightgbm'],
)

busqueda = RandomizedSearchCV(
    estimator=base,
    param_distributions=GRID_LGBM_EXT,
    n_iter=12,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=1,
    verbose=2,
    refit=True,
)
busqueda.fit(X_busqueda, y_busqueda)

print('Mejor F1-Macro CV:', round(busqueda.best_score_, 4))
print('Mejores parámetros:', busqueda.best_params_)

resultados_cv = (
    pd.DataFrame(busqueda.cv_results_)
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)
resultados_cv.to_csv(REPORTS_DIR / 'tuning_lightgbm_ext.csv', index=False)
resultados_cv.head(10)

In [ ]:
mejores = {
    clave.replace('modelo__', ''): valor
    for clave, valor in busqueda.best_params_.items()
}
print('Parámetros sin prefijo:', mejores)

modelo_busqueda = construir_lgbm_features(X_train, imputar=True, **mejores)
modelo_busqueda.fit(X_train, y_train)
met_busqueda = evaluar_modelo_ordinal(y_val, modelo_busqueda.predict(X_val))
print('Validación (mejor búsqueda):', {k: round(v, 4) for k, v in met_busqueda.items() if k != 'matriz_confusion'})

In [ ]:
variantes = []
configuraciones = [
    ('onehot', {}),
    (
        'target_encoding',
        {
            'target_encoding': [
                'neighbourhood_cleansed',
                'property_type',
                'neighbourhood',
            ]
        },
    ),
    ('onehot_agrupado', {'min_frequency': 0.01}),
    ('class_weight', {'class_weight': 'balanced'}),
]

for nombre, extra in configuraciones:
    print('=' * 70)
    print('Variante:', nombre)
    modelo = construir_lgbm_features(
        X_train,
        imputar=True,
        **mejores,
        **extra,
    )
    modelo.fit(X_train, y_train)
    met = evaluar_modelo_ordinal(y_val, modelo.predict(X_val))
    print({k: round(v, 4) for k, v in met.items() if k != 'matriz_confusion'})
    variantes.append(
        metricas_a_fila(
            f'lightgbm_ext_{nombre}',
            met,
            escenario='extendido_tuning',
            **{k: str(v) for k, v in extra.items()},
        )
    )

variantes_df = pd.DataFrame(variantes).sort_values('f1_macro', ascending=False)
variantes_df.to_csv(REPORTS_DIR / 'tuning_variantes_ext.csv', index=False)
display(variantes_df)

In [ ]:
mejor_variante = variantes_df.iloc[0]
nombre_variante = mejor_variante['modelo'].replace('lightgbm_ext_', '')
extra = dict(configuraciones[[c[0] for c in configuraciones].index(nombre_variante)][1])
print('Mejor variante:', nombre_variante)
print('Configuración extra:', extra)

modelo_final = construir_lgbm_features(
    X_train,
    imputar=True,
    **mejores,
    **extra,
)
modelo_final.fit(X_train, y_train)
met_test = evaluar_modelo_ordinal(y_test, modelo_final.predict(X_test))
print('Test:', {k: round(v, 4) for k, v in met_test.items() if k != 'matriz_confusion'})

pd.DataFrame(
    [
        metricas_a_fila(
            'lightgbm_ext_tuned',
            met_test,
            escenario='extendido_tuning',
            variante=nombre_variante,
            **mejores,
        )
    ]
).to_csv(REPORTS_DIR / 'lightgbm_ext_tuned_metricas.csv', index=False)

ConfusionMatrixDisplay(
    met_test['matriz_confusion'], display_labels=CLASES
).plot(cmap='Blues', values_format='d')
plt.title('LightGBM extendido + tuning - test')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    REPORTS_DIR / 'lightgbm_ext_tuned_matriz_confusion.png',
    dpi=150,
    bbox_inches='tight',
)
plt.show()

ruta = guardar_modelo(modelo_final, 'lightgbm_ext_tuned')
print('Modelo guardado en:', ruta)

config = {
    'modelo': 'lightgbm_ext_tuned',
    'variante': nombre_variante,
    'params': mejores,
    'extra': extra,
    'f1_macro_cv': float(busqueda.best_score_),
    'f1_macro_val': float(mejor_variante['f1_macro']),
    'accuracy_test': float(met_test['accuracy']),
    'f1_macro_test': float(met_test['f1_macro']),
    'accuracy_adjacent_test': float(met_test['accuracy_adjacent']),
}
with open(REPORTS_DIR / 'mejor_config_lightgbm_ext.json', 'w', encoding='utf-8') as archivo:
    json.dump(config, archivo, indent=2, ensure_ascii=False)
print(json.dumps(config, indent=2, ensure_ascii=False))

## Lectura de resultados

- `tuning_lightgbm_ext.csv` contiene todas las combinaciones evaluadas y su ranking.
- `tuning_variantes_ext.csv` compara One-Hot, target encoding y `class_weight`. Si el target encoding no mejora, se documenta y se conserva One-Hot: es más simple e interpretable.
- `lightgbm_ext_tuned_metricas.csv` es el resultado final del mejor esfuerzo para el 90 %.
- El test no participó en ninguna selección. La comparación con `lightgbm_ext` (configuración fija) y con `lightgbm_features` (geo) cuantifica la ganancia real del tuning.